In [17]:
import pandas as pd
import numpy as np
from math import ceil
from collections import defaultdict
from itertools import combinations

In [5]:
RUTA_DATASET = "dataset/data_secretariado.csv"

In [9]:
df = pd.read_csv(RUTA_DATASET, dtype=str)

print("Dimensiones del dataset:")
print(df.shape)


Dimensiones del dataset:
(133887, 11)


# **Ejercicio 1:** Implementacion Apriori

## **Preparación de los datos para Apriori**

El dataset ya viene limpio, por lo que no se realizará una limpieza profunda.
Sin embargo, sí se requieren algunas transformaciones para que los datos sean compatibles
con el algoritmo Apriori.

### Transformaciones realizadas

- **`ID_VICTIMA` se elimina** porque es un identificador único por registro; incluirlo
  generaría ítems que nunca se repiten y no aportaría ningún patrón útil.

- **Las fechas se discretizan por sexenio**. Usar la fecha completa (`2025-09-28 4:40:00`)
  generaría miles de valores únicos, haciendo imposible encontrar patrones frecuentes.
  Agrupar por sexenio permite capturar tendencias políticas y administrativas relevantes
  en el contexto mexicano, donde los cambios de gobierno pueden estar asociados a
  variaciones en el registro y atención de casos de desaparición.

- **Cada fila se convierte en una transacción**, es decir, un conjunto de ítems del tipo
  `COLUMNA=valor` (por ejemplo: `SEXO=MUJER`, `MES_DESAPARICION=9`, `ENTIDAD=PUEBLA`).

In [8]:
data = df.copy()

# ─── Parámetro de control ────────────────────────────────────────────────────
# Si es False, se eliminan filas donde las columnas clave sean CONFIDENCIAL.
# Esto reduce el dataset pero mejora la calidad de las reglas obtenidas,
# ya que los registros confidenciales aportan muy poca información útil.
INCLUIR_CONFIDENCIALES = False

if not INCLUIR_CONFIDENCIALES:
    columnas_confidenciales = ["SEXO", "ESTATUS_VICTIMA", "MUNICIPIO"]
    for col in columnas_confidenciales:
        data = data[data[col] != "CONFIDENCIAL"]

# ─── Conversión de fechas ─────────────────────────────────────────────────────
# Convertimos las columnas de fecha a tipo datetime.
# errors="coerce" convierte valores inválidos o vacíos en NaT (Not a Time),
# lo que nos permite manejarlos de forma controlada más adelante.
for col_str, col_dt in [
    ("FECHA_DESAPARICION", "FECHA_DESAPARICION_DT"),
    ("FECHA_REGISTRO",     "FECHA_REGISTRO_DT"),
    ("FECHA_NACIMIENTO",   "FECHA_NACIMIENTO_DT"),
]:
    data[col_dt] = pd.to_datetime(data[col_str], errors="coerce")

# ─── Discretización por sexenio ───────────────────────────────────────────────
# Usar la fecha completa generaría demasiados valores únicos, lo que haría
# imposible encontrar patrones frecuentes. Agrupar por sexenio permite capturar
# tendencias político-administrativas relevantes en el contexto mexicano,
# donde los cambios de gobierno pueden asociarse a variaciones en el registro
# y atención de casos de desaparición.

SEXENIOS_BINS   = [1994, 2000, 2006, 2012, 2018, 2024, 2030]
SEXENIOS_LABELS = [
    "Zedillo (1994-2000)",
    "Fox (2000-2006)",
    "Calderón (2006-2012)",
    "Peña Nieto (2012-2018)",
    "AMLO (2018-2024)",
    "Sheinbaum (2024-2030)",
]

def asignar_sexenio(serie_dt):
    """Recibe una Serie datetime y devuelve el sexenio correspondiente."""
    anio = serie_dt.dt.year
    return (
        pd.cut(anio, bins=SEXENIOS_BINS, labels=SEXENIOS_LABELS, right=False)
        .astype(str)
        .replace("nan", "DESCONOCIDO")
    )

data["SEXENIO_DESAPARICION"] = asignar_sexenio(data["FECHA_DESAPARICION_DT"])
data["SEXENIO_REGISTRO"]     = asignar_sexenio(data["FECHA_REGISTRO_DT"])

# ─── Mes de desaparición ──────────────────────────────────────────────────────
# Mantenemos el mes para detectar posibles patrones de estacionalidad,
# algo que el sexenio por sí solo no captura.
data["MES_DESAPARICION"] = (
    data["FECHA_DESAPARICION_DT"]
    .dt.month
    .astype("Int64")
    .astype(str)
    .str.zfill(2)
    .replace("<NA>", "DESCONOCIDO")
)

# ─── Grupo de edad ────────────────────────────────────────────────────────────
# Calculamos la edad aproximada al momento de la desaparición.
# Dividimos entre 365.25 para considerar los años bisiestos.
edad = (data["FECHA_DESAPARICION_DT"] - data["FECHA_NACIMIENTO_DT"]).dt.days / 365.25

# Descartamos edades fuera de un rango biológicamente plausible.
edad = edad.where((edad >= 0) & (edad <= 120))

bins_edad   = [-np.inf, 11, 17, 29, 44, 59, np.inf]
labels_edad = ["0-11", "12-17", "18-29", "30-44", "45-59", "60+"]
data["GRUPO_EDAD"] = (
    pd.cut(edad, bins=bins_edad, labels=labels_edad)
    .astype(str)
    .replace("nan", "DESCONOCIDO")
)

# ─── Selección de columnas para Apriori ───────────────────────────────────────
# Solo incluimos variables categóricas o ya discretizadas.
# Se excluye ORIGEN_REPORTE porque está altamente correlacionado con ENTIDAD
# y generaría reglas triviales del tipo "reporte de Puebla → entidad Puebla".
# Se excluyen las fechas crudas y los IDs.
COLUMNAS_ITEMS = [
    "SEXO",
    "ESTATUS_VICTIMA",
    "ENTIDAD",
    "MUNICIPIO",
    "SEXENIO_DESAPARICION",
    "SEXENIO_REGISTRO",
    "MES_DESAPARICION",
    "GRUPO_EDAD",
]

data_apriori = data[COLUMNAS_ITEMS].copy()

print("Dimensiones después de preparar los datos:")
print(data_apriori.shape)
data_apriori.head()

Dimensiones después de preparar los datos:
(84738, 8)


,SEXO,ESTATUS_VICTIMA,ENTIDAD,MUNICIPIO,SEXENIO_DESAPARICION,SEXENIO_REGISTRO,MES_DESAPARICION,GRUPO_EDAD
1,HOMBRE,DESAPARECIDA,SINALOA,CONCORDIA,Sheinbaum (2024-2030),Sheinbaum (2024-2030),09,45-59
5,HOMBRE,DESAPARECIDA,PUEBLA,PUEBLA,Sheinbaum (2024-2030),Sheinbaum (2024-2030),09,0-11
7,MUJER,DESAPARECIDA,PUEBLA,PUEBLA,Sheinbaum (2024-2030),Sheinbaum (2024-2030),09,30-44
11,HOMBRE,DESAPARECIDA,MICHOACÁN,TINGÜINDÍN,AMLO (2018-2024),AMLO (2018-2024),03,30-44
14,HOMBRE,DESAPARECIDA,HIDALGO,PACHUCA DE SOTO,Sheinbaum (2024-2030),Sheinbaum (2024-2030),09,0-11


Notemos que algunas reglas son esperadas debido a relaciones jerárquicas entre variables, por ejemplo MUNICIPIO y ENTIDAD.

In [13]:
def crear_transacciones(df_items):
    """
    Convierte un DataFrame en una lista de transacciones para Apriori.

    En minería de reglas de asociación, una "transacción" es un conjunto de ítems.
    Cada fila del DataFrame se convierte en una lista de ítems con el formato
    COLUMNA=valor (por ejemplo: SEXO=MUJER, ENTIDAD=PUEBLA).

    Parámetros
    ----------
    df_items : pd.DataFrame
        DataFrame donde cada fila es un registro y cada columna es un atributo.

    Retorna
    -------
    list[list]
        Lista de transacciones, una por fila del DataFrame.
    """
    transacciones = []

    for _, fila in df_items.iterrows():
        transaccion = set()

        for columna, valor in fila.items():
            # Ignoramos valores nulos, NaN o cadenas vacías, ya que no
            # aportan información y ensuciarían las reglas generadas.
            if pd.notna(valor) and str(valor).strip() not in ("", "nan"):
                item = f"{columna}={str(valor).strip()}"
                transaccion.add(item)

        transacciones.append(frozenset(transaccion))

    return transacciones

In [15]:
transacciones = crear_transacciones(data_apriori)

print(f"Número de transacciones: {len(transacciones)}")
print("\nEjemplo de transacción (índice 45):")
transacciones[45]

Número de transacciones: 84738

Ejemplo de transacción (índice 45):


frozenset({'ENTIDAD=TAMAULIPAS',
           'ESTATUS_VICTIMA=DESAPARECIDA',
           'GRUPO_EDAD=18-29',
           'MES_DESAPARICION=09',
           'MUNICIPIO=REYNOSA',
           'SEXENIO_DESAPARICION=Sheinbaum (2024-2030)',
           'SEXENIO_REGISTRO=Sheinbaum (2024-2030)',
           'SEXO=HOMBRE'})

## **Implementación de Apriori**

Apriori es un algoritmo clásico para encontrar **itemsets frecuentes**, es decir,
conjuntos de ítems que aparecen juntos con una frecuencia mínima en las transacciones.
Su nombre viene del principio *a priori*: **todo subconjunto de un itemset frecuente
también debe ser frecuente**. Esto permite podar el espacio de búsqueda y evitar
evaluar candidatos que de antemano sabemos que no cumplirán el umbral mínimo de soporte.

### Soporte

El soporte mide qué tan frecuente es un itemset en el dataset:

$$\text{soporte}(X) = \frac{\text{transacciones que contienen } X}{\text{total de transacciones}}$$

Solo se conservan los itemsets cuyo soporte sea mayor o igual al **soporte mínimo**
(`min_support`), que es un parámetro definido por el usuario.

### Pasos del algoritmo

1. Calcular el soporte de todos los ítems individuales y conservar los **frecuentes** (tamaño 1).
2. Generar **candidatos** de tamaño $k+1$ combinando los itemsets frecuentes de tamaño $k$.
3. Calcular el soporte de cada candidato.
4. Conservar únicamente los candidatos que superen el soporte mínimo.
5. Repetir los pasos 2–4 incrementando $k$ en cada iteración.
6. Detenerse cuando no existan nuevos candidatos frecuentes.

### **Funciones auxiliares**

In [19]:
def calcular_soporte_absoluto(itemset, transacciones):
    """
    Calcula el soporte absoluto de un itemset.

    Recorre todas las transacciones y cuenta en cuántas de ellas
    el itemset está contenido como subconjunto.

    Parámetros
    ----------
    itemset : frozenset
        Conjunto de ítems cuyo soporte se desea calcular.
    transacciones : list[frozenset]
        Lista de transacciones del dataset.

    Retorna
    -------
    int
        Número de transacciones que contienen al itemset.
    """
    return sum(1 for transaccion in transacciones if itemset.issubset(transaccion))


def obtener_itemsets_frecuentes_1(transacciones, min_soporte_absoluto):
    """def apriori(transacciones, min_support=0.02, max_k=3, mostrar_proceso=True):
    """
    Implementación del algoritmo Apriori para minería de itemsets frecuentes.

    Encuentra todos los itemsets cuyo soporte relativo supera el umbral
    `min_support`, hasta un tamaño máximo de `max_k` ítems por itemset.

    Parámetros
    ----------
    transacciones : list[frozenset]
        Lista de transacciones del dataset.
    min_support : float o int
        Soporte mínimo. Si está entre 0 y 1 se interpreta como fracción
        relativa (por ejemplo 0.02 = 2%). Si es >= 1 se interpreta como
        conteo absoluto.
    max_k : int o None
        Tamaño máximo de itemset a generar. Si es None, no hay límite.
    mostrar_proceso : bool
        Si es True, imprime el progreso de cada nivel k.

    Retorna
    -------
    soportes : dict[frozenset, float]
        Soporte relativo de cada itemset frecuente.
    conteos : dict[frozenset, int]
        Soporte absoluto de cada itemset frecuente.
    frecuentes_por_nivel : dict[int, dict[frozenset, int]]
        Itemsets frecuentes agrupados por tamaño k.
    """
    n = len(transacciones)
    min_soporte_absoluto = ceil(min_support * n) if 0 < min_support < 1 else int(min_support)

    soportes              = {}
    conteos               = {}
    frecuentes_por_nivel  = {}

    if mostrar_proceso:
        print(f"Número de transacciones:  {n}")
        print(f"Soporte mínimo absoluto:  {min_soporte_absoluto}")
        print(f"Soporte mínimo relativo:  {min_soporte_absoluto/n:.4f}")
        print()

    # ── Nivel 1: itemsets frecuentes de tamaño 1 ─────────────────────────────
    k  = 1
    Fk = obtener_itemsets_frecuentes_1(transacciones, min_soporte_absoluto)
    frecuentes_por_nivel[k] = Fk

    for itemset, conteo in Fk.items():
        conteos[itemset]   = conteo
        soportes[itemset]  = conteo / n

    if mostrar_proceso:
        print(f"F{k}: {len(Fk)} itemsets frecuentes")

    # ── Niveles k > 1: generar candidatos, podar y filtrar ───────────────────
    while Fk:
        k += 1
        if max_k is not None and k > max_k:
            break

        Ck = generar_candidatos(Fk.keys(), k)
        if mostrar_proceso:
            print(f"C{k}: {len(Ck)} candidatos generados")

        if not Ck:
            break

        Fk = filtrar_candidatos_por_soporte(Ck, transacciones, min_soporte_absoluto)
        if mostrar_proceso:
            print(f"F{k}: {len(Fk)} itemsets frecuentes")

        if not Fk:
            break

        frecuentes_por_nivel[k] = Fk
        for itemset, conteo in Fk.items():
            conteos[itemset]  = conteo
            soportes[itemset] = conteo / n

    return soportes, conteos, frecuentes_por_nivel
    Obtiene los itemsets frecuentes de tamaño 1 (F1).

    Recorre todas las transacciones contando las ocurrencias de cada
    ítem individual. Solo se conservan los ítems cuyo conteo supere
    el soporte mínimo absoluto.

    Parámetros
    ----------
    transacciones : list[frozenset]
        Lista de transacciones del dataset.
    min_soporte_absoluto : int
        Número mínimo de transacciones en las que debe aparecer un ítem
        para considerarse frecuente.

    Retorna
    -------
    dict[frozenset, int]
        Diccionario que mapea cada itemset frecuente de tamaño 1
        a su conteo absoluto.
    """
    conteos = defaultdict(int)
    for transaccion in transacciones:
        for item in transaccion:
            conteos[frozenset([item])] += 1

    return {
        itemset: conteo
        for itemset, conteo in conteos.items()
        if conteo >= min_soporte_absoluto
    }


def tiene_subconjuntos_frecuentes(candidato, frecuentes_anteriores):
    """
    Verifica la propiedad Apriori (poda por subconjuntos).

    Un candidato de tamaño k solo puede ser frecuente si **todos** sus
    subconjuntos de tamaño k-1 también lo son. Si alguno no aparece en
    los frecuentes anteriores, el candidato se descarta de inmediato,
    ahorrando el cálculo de su soporte.

    Parámetros
    ----------
    candidato : frozenset
        Itemset candidato de tamaño k.
    frecuentes_anteriores : iterable[frozenset]
        Itemsets frecuentes de tamaño k-1.

    Retorna
    -------
    bool
        True si todos los subconjuntos de tamaño k-1 son frecuentes,
        False en cuanto se encuentre uno que no lo sea.
    """
    k = len(candidato)
    frecuentes_anteriores = set(frecuentes_anteriores)
    for subconjunto in combinations(candidato, k - 1):
        if frozenset(subconjunto) not in frecuentes_anteriores:
            return False
    return True


def generar_candidatos(frecuentes_anteriores, k):
    """
    Genera candidatos de tamaño k a partir de los itemsets frecuentes de tamaño k-1.

    Se combinan pares de itemsets frecuentes mediante unión de conjuntos.
    Solo se conservan los candidatos que:
      1. Tienen exactamente k ítems.
      2. Pasan la poda Apriori (todos sus subconjuntos de tamaño k-1 son frecuentes).

    Parámetros
    ----------
    frecuentes_anteriores : iterable[frozenset]
        Itemsets frecuentes de tamaño k-1.
    k : int
        Tamaño deseado para los candidatos generados.

    Retorna
    -------
    set[frozenset]
        Conjunto de candidatos que superaron la poda Apriori.
    """
    candidatos = set()
    frecuentes_anteriores = list(frecuentes_anteriores)

    for i in range(len(frecuentes_anteriores)):
        for j in range(i + 1, len(frecuentes_anteriores)):
            candidato = frecuentes_anteriores[i] | frecuentes_anteriores[j]
            if len(candidato) == k:
                if tiene_subconjuntos_frecuentes(candidato, frecuentes_anteriores):
                    candidatos.add(candidato)

    return candidatos


def filtrar_candidatos_por_soporte(candidatos, transacciones, min_soporte_absoluto):
    """
    Filtra los candidatos que superan el soporte mínimo absoluto.

    Para cada candidato se calcula su soporte absoluto recorriendo
    las transacciones. Solo se conservan los que cumplen el umbral.

    Parámetros
    ----------
    candidatos : set[frozenset]
        Candidatos a evaluar.
    transacciones : list[frozenset]
        Lista de transacciones del dataset.
    min_soporte_absoluto : int
        Umbral mínimo de soporte absoluto.

    Retorna
    -------
    dict[frozenset, int]
        Diccionario con los candidatos frecuentes y su conteo absoluto.
    """
    return {
        candidato: conteo
        for candidato in candidatos
        if (conteo := calcular_soporte_absoluto(candidato, transacciones))
        >= min_soporte_absoluto
    }

### **Implementación algoritmo**

In [22]:
def apriori(transacciones, min_support=0.02, max_k=3, mostrar_proceso=True):
    """
    Implementación del algoritmo Apriori para minería de itemsets frecuentes.

    Encuentra todos los itemsets cuyo soporte relativo supera el umbral
    `min_support`, hasta un tamaño máximo de `max_k` ítems por itemset.

    Parámetros
    ----------
    transacciones : list[frozenset]
        Lista de transacciones del dataset.
    min_support : float o int
        Soporte mínimo. Si está entre 0 y 1 se interpreta como fracción
        relativa (por ejemplo 0.02 = 2%). Si es >= 1 se interpreta como
        conteo absoluto.
    max_k : int o None
        Tamaño máximo de itemset a generar. Si es None, no hay límite.
    mostrar_proceso : bool
        Si es True, imprime el progreso de cada nivel k.

    Retorna
    -------
    soportes : dict[frozenset, float]
        Soporte relativo de cada itemset frecuente.
    conteos : dict[frozenset, int]
        Soporte absoluto de cada itemset frecuente.
    frecuentes_por_nivel : dict[int, dict[frozenset, int]]
        Itemsets frecuentes agrupados por tamaño k.
    """
    n = len(transacciones)
    min_soporte_absoluto = ceil(min_support * n) if 0 < min_support < 1 else int(min_support)

    soportes              = {}
    conteos               = {}
    frecuentes_por_nivel  = {}

    if mostrar_proceso:
        print(f"Número de transacciones:  {n}")
        print(f"Soporte mínimo absoluto:  {min_soporte_absoluto}")
        print(f"Soporte mínimo relativo:  {min_soporte_absoluto/n:.4f}")
        print()

    # ── Nivel 1: itemsets frecuentes de tamaño 1 ─────────────────────────────
    k  = 1
    Fk = obtener_itemsets_frecuentes_1(transacciones, min_soporte_absoluto)
    frecuentes_por_nivel[k] = Fk

    for itemset, conteo in Fk.items():
        conteos[itemset]   = conteo
        soportes[itemset]  = conteo / n

    if mostrar_proceso:
        print(f"F{k}: {len(Fk)} itemsets frecuentes")

    # ── Niveles k > 1: generar candidatos, podar y filtrar ───────────────────
    while Fk:
        k += 1
        if max_k is not None and k > max_k:
            break

        Ck = generar_candidatos(Fk.keys(), k)
        if mostrar_proceso:
            print(f"C{k}: {len(Ck)} candidatos generados")

        if not Ck:
            break

        Fk = filtrar_candidatos_por_soporte(Ck, transacciones, min_soporte_absoluto)
        if mostrar_proceso:
            print(f"F{k}: {len(Fk)} itemsets frecuentes")

        if not Fk:
            break

        frecuentes_por_nivel[k] = Fk
        for itemset, conteo in Fk.items():
            conteos[itemset]  = conteo
            soportes[itemset] = conteo / n

    return soportes, conteos, frecuentes_por_nivel

### **Ejecución de apriori**

In [23]:
MIN_SUPPORT = 0.02
MAX_K = 3

soportes, conteos, frecuentes_por_nivel = apriori(
    transacciones,
    min_support=MIN_SUPPORT,
    max_k=MAX_K,
    mostrar_proceso=True
)

print()
print("Total de itemsets frecuentes encontrados:", len(soportes))

print("\nItemsets frecuentes por tamaño:")
for k, itemsets in frecuentes_por_nivel.items():
    print(f"Tamaño {k}: {len(itemsets)}")

Número de transacciones:  84738
Soporte mínimo absoluto:  1695
Soporte mínimo relativo:  0.0200

F1: 53 itemsets frecuentes
C2: 1378 candidatos generados
F2: 201 itemsets frecuentes
C3: 330 candidatos generados
F3: 228 itemsets frecuentes

Total de itemsets frecuentes encontrados: 482

Itemsets frecuentes por tamaño:
Tamaño 1: 53
Tamaño 2: 201
Tamaño 3: 228


### **Observaciones sobre los itemsets frecuentes encontrados**

Con un soporte mínimo del 2% (al menos 1,695 transacciones), el algoritmo encontró
**482 itemsets frecuentes** en total. Algunos puntos a destacar:

- **F1 (53 itemsets):** Hay 53 ítems individuales que aparecen en al menos el 2% de
  las transacciones. Esto indica que el dataset tiene una diversidad moderada de valores
  frecuentes.

- **F2 (201 itemsets):** De los 1,378 candidatos de tamaño 2 generados, solo 201
  superaron el umbral. La poda Apriori eliminó el 85% de los candidatos antes de
  calcular su soporte, lo que demuestra la eficiencia del principio *a priori*.

- **F3 (228 itemsets) > F2 (201 itemsets):** Es llamativo que haya más itemsets
  frecuentes de tamaño 3 que de tamaño 2. Esto sugiere que ciertas variables están
  fuertemente correlacionadas entre sí — por ejemplo, `ENTIDAD`, `MUNICIPIO` y
  `SEXENIO_DESAPARICION` tienden a aparecer juntas de forma consistente, lo que
  produce muchas combinaciones frecuentes de tamaño 3 a partir de un número
  relativamente menor de pares frecuentes.

In [24]:
def itemset_a_texto(itemset):
    # Convierte un itemset en texto para mostrarlo en tablas.
    return ", ".join(sorted(itemset))

itemsets_frecuentes = []

for itemset, soporte in soportes.items():
    itemsets_frecuentes.append({
        "itemset": itemset_a_texto(itemset),
        "tamaño": len(itemset),
        "conteo_soporte": conteos[itemset],
        "soporte": soporte
    })

df_itemsets_frecuentes = pd.DataFrame(itemsets_frecuentes)

df_itemsets_frecuentes = df_itemsets_frecuentes.sort_values(
    by=["tamaño", "soporte"],
    ascending=[True, False]
)

df_itemsets_frecuentes

,itemset,tamaño,conteo_soporte,soporte
5,ESTATUS_VICTIMA=DESAPARECIDA,1,79881,0.942682
4,SEXO=HOMBRE,1,64666,0.763129
11,SEXENIO_REGISTRO=AMLO (2018-2024),1,29181,0.344367
13,SEXENIO_DESAPARICION=AMLO (2018-2024),1,26265,0.309955
24,SEXENIO_DESAPARICION=Peña Nieto (2012-2018),1,22927,0.270563
...,...,...,...,...
385,"ENTIDAD=SE DESCONOCE, MUNICIPIO=SE DESCONOCE, ...",3,1719,0.020286
448,"ENTIDAD=TAMAULIPAS, ESTATUS_VICTIMA=DESAPARECI...",3,1713,0.020215
266,"ENTIDAD=SONORA, SEXENIO_REGISTRO=AMLO (2018-20...",3,1707,0.020144
293,"ENTIDAD=SONORA, SEXENIO_DESAPARICION=AMLO (201...",3,1706,0.020133


## **Generación de reglas de asociación**

A partir de los itemsets frecuentes se generan **reglas de asociación**, que expresan
relaciones del tipo:

$$\text{antecedente} \Rightarrow \text{consecuente}$$

Por ejemplo:

$$\text{SEXO=MUJER, GRUPO\_EDAD=12-17} \Rightarrow \text{ENTIDAD=ESTADO DE MÉXICO}$$

Esta regla se leería como: *"cuando la víctima es mujer y tiene entre 12 y 17 años,
es frecuente que la desaparición haya ocurrido en el Estado de México"*.

### Medidas de interés

Para determinar si una regla es útil o trivial, se evalúan tres métricas:

**Soporte**
Mide qué tan frecuente es la regla en el dataset. Un soporte alto indica que la
combinación antecedente + consecuente aparece en una porción significativa de los casos.

$$\text{soporte}(X \Rightarrow Y) = \frac{|\text{transacciones que contienen } X \cup Y|}{|\text{total de transacciones}|}$$

**Confianza**
Mide la fiabilidad de la regla: dado que ocurre el antecedente, ¿qué tan probable
es que ocurra el consecuente?

$$\text{confianza}(X \Rightarrow Y) = \frac{\text{soporte}(X \cup Y)}{\text{soporte}(X)}$$

**Lift**
Mide si la asociación entre antecedente y consecuente es real o simplemente producto
de que ambos son frecuentes por separado.

$$\text{lift}(X \Rightarrow Y) = \frac{\text{confianza}(X \Rightarrow Y)}{\text{soporte}(Y)}$$

| Valor de lift | Interpretación |
|---|---|
| $> 1$ | Asociación positiva: el antecedente favorece al consecuente |
| $= 1$ | Independencia: no hay asociación real |
| $< 1$ | Asociación negativa: el antecedente inhibe al consecuente |

In [27]:
def generar_reglas(soportes, conteos, min_confidence=0.70):
    """
    Genera reglas de asociación a partir de los itemsets frecuentes.

    Para cada itemset frecuente de tamaño >= 2, se generan todas las
    particiones posibles en antecedente y consecuente. Una regla se
    conserva únicamente si su confianza supera el umbral `min_confidence`.

    El proceso para cada itemset I es:
      1. Enumerar todos sus subconjuntos no vacíos como posibles antecedentes.
      2. El consecuente es el complemento: I - antecedente.
      3. Calcular confianza = soporte(I) / soporte(antecedente).
      4. Calcular lift = confianza / soporte(consecuente).
      5. Conservar la regla si confianza >= min_confidence.

    Nota: solo se pueden generar reglas para itemsets cuyos antecedentes
    y consecuentes también sean itemsets frecuentes (es decir, estén en
    `soportes`). Esto está garantizado por la propiedad Apriori.

    Parámetros
    ----------
    soportes : dict[frozenset, float]
        Soporte relativo de cada itemset frecuente.
    conteos : dict[frozenset, int]
        Soporte absoluto de cada itemset frecuente.
    min_confidence : float
        Confianza mínima para conservar una regla (entre 0 y 1).

    Retorna
    -------
    pd.DataFrame
        DataFrame con las reglas que superaron el umbral de confianza,
        ordenadas de mayor a menor por confianza, lift y soporte.
        Columnas: antecedente, consecuente, conteo_soporte, soporte,
        confianza, lift.
    """
    reglas = []

    for itemset, soporte_itemset in soportes.items():
        # Solo procesamos itemsets de tamaño >= 2, ya que una regla
        # necesita al menos un ítem en el antecedente y uno en el consecuente.
        if len(itemset) < 2:
            continue

        items = list(itemset)

        # Generamos todas las particiones antecedente → consecuente.
        # r va de 1 a len-1 para garantizar que ambos lados sean no vacíos.
        for r in range(1, len(items)):
            for antecedente_tuple in combinations(items, r):
                antecedente = frozenset(antecedente_tuple)
                consecuente = itemset - antecedente

                soporte_antecedente = soportes.get(antecedente)
                soporte_consecuente = soportes.get(consecuente)

                # Si alguno de los dos no está en los frecuentes, no podemos
                # calcular las métricas con precisión; descartamos la regla.
                if soporte_antecedente is None or soporte_consecuente is None:
                    continue

                confianza = soporte_itemset / soporte_antecedente
                lift      = confianza / soporte_consecuente

                if confianza >= min_confidence:
                    reglas.append({
                        "antecedente"    : itemset_a_texto(antecedente),
                        "consecuente"    : itemset_a_texto(consecuente),
                        "conteo_soporte" : conteos[itemset],
                        "soporte"        : soporte_itemset,
                        "confianza"      : confianza,
                        "lift"           : lift,
                    })

    columnas = ["antecedente", "consecuente", "conteo_soporte", "soporte", "confianza", "lift"]
    df_reglas = pd.DataFrame(reglas, columns=columnas)

    if not df_reglas.empty:
        df_reglas = df_reglas.sort_values(
            by=["confianza", "lift", "soporte"],
            ascending=False
        ).reset_index(drop=True)

    return df_reglas

In [28]:
MIN_CONFIDENCE = 0.70

reglas = generar_reglas(
    soportes,
    conteos,
    min_confidence=MIN_CONFIDENCE
)

print("Total de reglas fuertes encontradas:", len(reglas))

reglas

Total de reglas fuertes encontradas: 490


,antecedente,consecuente,conteo_soporte,soporte,confianza,lift
0,MUNICIPIO=TIJUANA,ENTIDAD=BAJA CALIFORNIA,2064,0.024357,1.000000,28.199002
1,"ESTATUS_VICTIMA=DESAPARECIDA, MUNICIPIO=TIJUANA",ENTIDAD=BAJA CALIFORNIA,2051,0.024204,1.000000,28.199002
2,MUNICIPIO=CULIACÁN,ENTIDAD=SINALOA,1703,0.020097,1.000000,17.991083
3,ENTIDAD=SE DESCONOCE,MUNICIPIO=SE DESCONOCE,2369,0.027957,1.000000,15.703855
4,"ENTIDAD=SE DESCONOCE, ESTATUS_VICTIMA=DESAPARE...",MUNICIPIO=SE DESCONOCE,2349,0.027721,1.000000,15.703855
...,...,...,...,...,...,...
485,"ESTATUS_VICTIMA=DESAPARECIDA, MUNICIPIO=SE DES...",SEXO=HOMBRE,3552,0.041917,0.707993,0.927750
486,"ENTIDAD=TABASCO, ESTATUS_VICTIMA=DESAPARECIDA",SEXENIO_DESAPARICION=AMLO (2018-2024),1838,0.021690,0.707740,2.283360
487,ENTIDAD=NUEVO LEÓN,"ESTATUS_VICTIMA=DESAPARECIDA, SEXO=HOMBRE",2582,0.030470,0.704118,0.981180
488,ENTIDAD=TABASCO,"ESTATUS_VICTIMA=DESAPARECIDA, SEXENIO_REGISTRO...",1842,0.021738,0.701982,2.250218


### **Observaciones sobre las reglas generadas**

Con una confianza mínima del 70%, el algoritmo encontró **490 reglas de asociación**.
A continuación se destacan los patrones más relevantes:

### Reglas triviales geográficas

Las reglas con confianza = 1.0 encabezando la tabla son del tipo:

$$\text{MUNICIPIO=TIJUANA} \Rightarrow \text{ENTIDAD=BAJA CALIFORNIA}$$

Estas reglas son **matemáticamente válidas pero informativamente vacías**: un municipio
siempre pertenece a la misma entidad, por lo que la asociación es trivial por definición
geográfica. Su lift extremadamente alto (28.19) no refleja un patrón interesante, sino
simplemente esta dependencia estructural entre variables.

### Sesgo en el registro de datos

La regla:

$$\text{ENTIDAD=SE DESCONOCE} \Rightarrow \text{MUNICIPIO=SE DESCONOCE}$$

tiene confianza = 1.0 y es otro caso trivial. Sin embargo, sí revela algo importante:
**la falta de información en una columna arrastra la ausencia de información en otras**,
lo que sugiere un sesgo sistemático en el proceso de registro de casos.

### Reglas potencialmente interesantes

Las reglas hacia el final de la tabla, con confianza en torno al 70%, capturan patrones
más significativos. Por ejemplo:

$$\text{ENTIDAD=TABASCO} \Rightarrow \text{ESTATUS\_VICTIMA=DESAPARECIDA, SEXENIO=AMLO}$$

Este tipo de reglas asocian una entidad federativa con un período de gobierno específico
y un estatus de víctima, lo que puede reflejar tendencias reales en el fenómeno de
desapariciones durante ciertos períodos administrativos.